In [ ]:
# pip installs

!pip install -q datasets requests torch peft bitsandbytes transformers trl accelerate sentencepiece wandb matplotlib

Giải thích: Lệnh trên cài đặt các thư viện cần thiết cho việc xử lý dữ liệu, huấn luyện mô hình, và theo dõi quá trình huấn luyện. Các thư viện bao gồm:

datasets: Xử lý dữ liệu.

requests: Gửi yêu cầu HTTP.

torch: Thư viện PyTorch cho học sâu.

peft: LoRA (Low-Rank Adaptation) cho việc fine-tune mô hình.

bitsandbytes: Thư viện hỗ trợ giảm độ chính xác để tối ưu bộ nhớ khi huấn luyện.

transformers: Thư viện của Hugging Face để làm việc với các mô hình ngôn ngữ.

trl: Thư viện hỗ trợ huấn luyện mô hình theo phương pháp SFT (Supervised Fine-Tuning).

accelerate: Tăng tốc độ huấn luyện.

sentencepiece: Công cụ mã hóa cho dữ liệu văn bản.

wandb: Thư viện theo dõi và giám sát quá trình huấn luyện.

matplotlib: Thư viện vẽ đồ thị

In [ ]:
# imports
# With much thanks to Islam S. for identifying that there was a missing import!

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

 Đoạn mã này nhập các thư viện và module cần thiết:

Các thư viện chuẩn của Python như os, re, math, và datetime.

tqdm: Hiển thị tiến độ của vòng lặp.

userdata: Quản lý thông tin người dùng trong Google Colab.

huggingface_hub: Để đăng nhập và làm việc với mô hình trên Hugging Face.

torch: Thư viện PyTorch.

transformers: Thư viện của Hugging Face để làm việc với mô hình ngôn ngữ.

datasets: Xử lý và tải dữ liệu.

wandb: Quản lý và theo dõi quá trình huấn luyện.

peft: LoRA.

trl: Thư viện cho Supervised Fine-Tuning (SFT)

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME = "pricer"
HF_USER = "ed-donner" # your HF name here!

# Data

DATASET_NAME = f"{HF_USER}/pricer-data"
# Or just use the one I've uploaded
# DATASET_NAME = "ed-donner/pricer-data"
MAX_SEQUENCE_LENGTH = 182

# Run name for saving the model in the hub

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyperparameters for QLoRA

LORA_R = 32
LORA_ALPHA = 64
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1
QUANT_4_BIT = True

# Hyperparameters for Training

EPOCHS = 1 # you can do more epochs if you wish, but only 1 is needed - more is probably overkill
BATCH_SIZE = 4 # on an A100 box this can go up to 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Admin config - note that SAVE_STEPS is how often it will upload to the hub
# I've changed this from 5000 to 2000 so that you get more frequent saves

STEPS = 50
SAVE_STEPS = 2000
LOG_TO_WANDB = True

%matplotlib inline

1. Cấu hình các hằng số (Constants):
BASE_MODEL: Tên của mô hình cơ sở mà bạn sẽ sử dụng cho việc fine-tune. Mô hình này là "Meta-Llama-3.1-8B" từ Meta.

PROJECT_NAME: Tên của dự án bạn đang thực hiện, ở đây là "pricer" (có thể liên quan đến dự đoán giá).

HF_USER: Tên người dùng của bạn trên Hugging Face. Bạn sẽ cần sử dụng tên này để đăng tải mô hình lên Hugging Face Hub.
2. Cấu hình bộ dữ liệu (Data):
DATASET_NAME: Tên bộ dữ liệu mà bạn sẽ sử dụng để huấn luyện mô hình. Đây là một chuỗi định dạng, kết hợp tên người dùng HF với tên bộ dữ liệu (ở đây là "pricer-data").

MAX_SEQUENCE_LENGTH: Chiều dài tối đa của mỗi chuỗi đầu vào khi huấn luyện. Bộ dữ liệu sẽ được cắt bớt để đảm bảo mỗi chuỗi có độ dài không vượt quá giá trị này.
3. Tên phiên huấn luyện (Run name) cho mô hình khi lưu trên Hugging Face Hub:
python

RUN_NAME: Tạo tên phiên huấn luyện bằng cách sử dụng ngày và giờ hiện tại, điều này giúp xác định được mỗi lần chạy huấn luyện khác nhau.

PROJECT_RUN_NAME: Kết hợp PROJECT_NAME và RUN_NAME để tạo thành tên phiên huấn luyện cho dự án.

HUB_MODEL_NAME: Tên mô hình sẽ được lưu lên Hugging Face Hub. Nó sẽ kết hợp tên người dùng (HF_USER) và tên phiên huấn luyện (PROJECT_RUN_NAME).
4. Cấu hình các tham số cho LoRA (QLoRA):
LORA_R: Số lượng các yếu tố hạng thấp trong mô hình LoRA. Đây là tham số quan trọng giúp điều chỉnh mức độ "nén" của mô hình.

LORA_ALPHA: Tham số điều chỉnh độ mạnh của các yếu tố LoRA.

TARGET_MODULES: Danh sách các module trong mô hình sẽ được áp dụng LoRA. Các module này liên quan đến các phép toán trong mô hình transformer (q_proj, v_proj, k_proj, o_proj).

LORA_DROPOUT: Tỷ lệ dropout áp dụng cho LoRA để tránh quá khớp (overfitting).

QUANT_4_BIT: Chỉ thị có sử dụng 4-bit quantization không. Nếu là True, mô hình sẽ sử dụng kỹ thuật 4-bit để giảm bộ nhớ và tăng tốc độ xử lý.
5. Cấu hình các tham số huấn luyện:
EPOCHS: Số vòng huấn luyện. Mỗi vòng là một lần chạy qua toàn bộ dữ liệu. Ở đây là 1 epoch.

BATCH_SIZE: Kích thước mỗi batch (lượng dữ liệu xử lý một lần). Ở đây là 4.

GRADIENT_ACCUMULATION_STEPS: Số bước tích lũy gradient trước khi cập nhật trọng số. Điều này giúp giảm bộ nhớ cần thiết.

LEARNING_RATE: Tỷ lệ học (learning rate), một tham số quan trọng điều chỉnh tốc độ học của mô hình.

LR_SCHEDULER_TYPE: Loại lịch trình giảm tỷ lệ học, ở đây sử dụng kiểu cosine, giúp tỷ lệ học giảm dần theo thời gian theo một hàm cosin.

WARMUP_RATIO: Tỷ lệ warm-up (khởi động) cho tỷ lệ học.

OPTIMIZER: Bộ tối ưu hóa, ở đây là paged_adamw_32bit, một loại Adam optimizer với độ chính xác 32-bit.
6. Cấu hình cho việc lưu mô hình và giám sát huấn luyện:
STEPS: Số bước huấn luyện trước khi ghi lại trạng thái huấn luyện (ví dụ, ghi log, lưu mô hình, v.v.).

SAVE_STEPS: Số bước sau đó mô hình sẽ được lưu lên Hugging Face Hub. Giá trị này là 2000 bước, có nghĩa là mô hình sẽ được lưu lên Hub sau mỗi 2000 bước huấn luyện.

LOG_TO_WANDB: Nếu True, quá trình huấn luyện sẽ được theo dõi và ghi lại trên Weights & Biases (wandb), giúp theo dõi các thông số như độ mất mát, độ chính xác, và các chỉ số khác trong quá trình huấn luyện.


In [ ]:
HUB_MODEL_NAME

In [ ]:
Tạo tên mô hình hoàn chỉnh để lưu trữ trên Hugging Face Model Hub.

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

hf_token = userdata.get('HF_TOKEN'): Lấy token từ userdata.

login(hf_token, add_to_git_credential=True): Đăng nhập vào Hugging Face với token, và thêm token vào thông tin xác thực Git.

In [ ]:
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

Đăng nhập vào Weights & Biases:

wandb_api_key = userdata.get('WANDB_API_KEY'): Lấy API key từ userdata để sử dụng trong đăng nhập.

os.environ["WANDB_API_KEY"] = wandb_api_key: Thiết lập API key cho Weights & Biases trong môi trường hệ thống.

wandb.login(): Đăng nhập vào W&B bằng API key.

Cấu hình Weights & Biases:

os.environ["WANDB_PROJECT"] = PROJECT_NAME: Thiết lập tên dự án trong W&B.

os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end": Cấu hình việc lưu trữ mô hình, nếu LOG_TO_WANDB là True, sẽ lưu checkpoint; nếu không, chỉ lưu mô hình cuối cùng.

os.environ["WANDB_WATCH"] = "gradients": Theo dõi gradient trong quá trình huấn luyện mô hình.











In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

Đoạn mã trên tải bộ dữ liệu từ Hugging Face, phân chia thành tập huấn luyện (train) và tập kiểm tra (test) để sử dụng trong quá trình huấn luyện và đánh giá mô hình.

In [ ]:
# Nếu bạn muốn giảm số điểm dữ liệu huấn luyện xuống còn 20,000 mẫu, bạn chỉ cần bỏ dấu # ở đầu dòng mã:
# train = train.select(range(20000))

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

Đoạn mã này khởi tạo một phiên làm việc trong Weights & Biases để theo dõi quá trình huấn luyện mô hình, với tên dự án và phiên làm việc được chỉ định từ các biến PROJECT_NAME và RUN_NAME.

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

Đoạn mã trên thiết lập cấu hình lượng tử hóa cho mô hình dựa trên tùy chọn QUANT_4_BIT. Nếu QUANT_4_BIT là True, nó sẽ sử dụng cấu hình lượng tử hóa 4-bit với các tham số bổ sung. Nếu không, nó sẽ sử dụng cấu hình lượng tử hóa 8-bit. Cấu hình này giúp giảm bộ nhớ cần thiết khi tải mô hình và có thể cải thiện hiệu suất khi huấn luyện trên các thiết bị hạn chế bộ nhớ.

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

Tải Tokenizer:

AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True): Tải tokenizer từ mô hình cơ sở đã chỉ định (BASE_MODEL), cho phép sử dụng mã nguồn từ xa.

tokenizer.pad_token = tokenizer.eos_token: Thiết lập token padding là token kết thúc chuỗi (eos token).

tokenizer.padding_side = "right": Đảm bảo rằng padding sẽ được thêm vào phía bên phải chuỗi.

Tải Model:

AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant_config, device_map="auto"): Tải mô hình học sâu cho các tác vụ ngôn ngữ tự nhiên từ mô hình cơ sở, với cấu hình lượng tử hóa và tự động phân phối mô hình lên các thiết bị (CPU hoặc GPU).

base_model.generation_config.pad_token_id = tokenizer.pad_token_id: Cấu hình lại ID token padding cho mô hình để nó sử dụng đúng token padding đã định nghĩa.

In thông tin bộ nhớ:

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB"): In ra dung lượng bộ nhớ của mô hình, được tính theo megabyte (MB).

In [ ]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "Price is $"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

Tạo Data Collator:

DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer): Khởi tạo một DataCollator cho mô hình hoàn thiện văn bản (language modeling), sử dụng mẫu câu response_template = "Price is $" và tokenizer đã được tải từ trước.

In [ ]:
# First, specify the configuration parameters for LoRA

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# Next, specify the general configuration parameters for training

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True
)

# And now, the Supervised Fine Tuning Trainer will carry out the fine-tuning
# Given these 2 sets of configuration parameters
# The latest version of trl is showing a warning about labels - please ignore this warning
# But let me know if you don't see good training results (loss coming down).

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    args=train_parameters,
    data_collator=collator
  )

Cấu hình LoRA (Low-Rank Adaptation):

LoraConfig được sử dụng để thiết lập các tham số LoRA, giúp mô hình có thể học được các điều chỉnh hiệu quả mà không cần thay đổi quá nhiều tham số gốc của mô hình.

Các tham số LoRA bao gồm:

lora_alpha, lora_dropout, r: Các tham số kiểm soát quá trình học LoRA.

bias="none": Không áp dụng bias cho các tham số LoRA.

task_type="CAUSAL_LM": Loại tác vụ là mô hình ngôn ngữ tuần tự (causal language modeling).

target_modules: Các module mà LoRA sẽ áp dụng.

Cấu hình tham số huấn luyện:

SFTConfig được sử dụng để cấu hình các tham số huấn luyện cho mô hình.

Các tham số bao gồm:

output_dir: Thư mục để lưu mô hình sau huấn luyện.

num_train_epochs: Số vòng lặp huấn luyện.

per_device_train_batch_size: Kích thước batch huấn luyện.

gradient_accumulation_steps: Số bước tích lũy gradient trước khi cập nhật.

learning_rate: Tốc độ học.

optimizer: Loại tối ưu hóa (ở đây là paged_adamw_32bit).

Các tham số khác như chiến lược lưu mô hình, chiến lược learning rate, và việc báo cáo lên Weights & Biases.

Huấn luyện mô hình:

SFTTrainer được sử dụng để khởi tạo và thực hiện huấn luyện mô hình với các tham số đã cấu hình ở trên.

Mô hình được huấn luyện với dữ liệu train và sử dụng collator để chuẩn bị dữ liệu đầu vào.

Tóm lại, đoạn mã này thiết lập các tham số cho LoRA và huấn luyện mô hình bằng cách sử dụng SFTTrainer.

In [ ]:
Cấu hình các tham số cho LoRA (Low-Rank Adaptation) để fine-tune mô hình.

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

In [ ]:
Giải thích: Sau khi huấn luyện xong, mô hình được đẩy lên Hugging Face Hub để lưu trữ và chia sẻ.

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()